In [1]:
!pip install lightning timm audiomentations
import os
import gc
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from torch.utils.data import DataLoader, Dataset
import timm
import torchmetrics
from pathlib import Path
from audiomentations import Compose, AddGaussianNoise, TimeStretch

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# Фіксуємо seed для відтворюваності експериментів
pl.seed_everything(42)

DATA_DIR = Path("/kaggle/input/competitions/birdclef-2026")
SAMPLE_RATE = 32000
CHUNK_LEN = 5 * SAMPLE_RATE

sub_df = pd.read_csv(DATA_DIR / 'sample_submission.csv')
BIRD_SPECIES = sub_df.columns[1:].tolist()
NUM_BIRDS = len(BIRD_SPECIES)

df_labels = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv").dropna(subset=['primary_label'])

# Фільтрація класів: залишаємо лише ті види птахів, які реально представлені у тренувальній вибірці
active_birds_set = {b.strip() for labels in df_labels['primary_label'] for b in str(labels).split(';')}
active_bird_indices = [i for i, b in enumerate(BIRD_SPECIES) if b in active_birds_set]
ACTIVE_BIRD_SPECIES = [BIRD_SPECIES[i] for i in active_bird_indices]
NUM_ACTIVE_BIRDS = len(ACTIVE_BIRD_SPECIES)

print(f"Птахів у тренувальній вибірці: {NUM_ACTIVE_BIRDS} / {NUM_BIRDS}")

# --- КАСТОМНА ФУНКЦІЯ ВТРАТ ---
class FocalLoss(torch.nn.Module):
    """
    Focal Loss використовується для вирішення проблеми дисбалансу класів (class imbalance).
    Вона динамічно масштабує Cross Entropy Loss, фокусуючи увагу моделі на складних 
    для розпізнавання (рідкісних) прикладах і зменшуючи вагу впевнених передбачень.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        probs = torch.sigmoid(inputs)
        
        p_t = probs * targets + (1 - probs) * (1 - targets)
        loss = bce_loss * ((1 - p_t) ** self.gamma)
        
        if self.alpha >= 0:
            alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
            loss = alpha_t * loss
            
        return loss.mean()

# --- ОБРОБКА ДАНИХ ТА АУГМЕНТАЦІЯ ---
class BirdCLEFDataset(Dataset):
    def __init__(self, df, data_dir, augment=False):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.augment = augment
        
        # Перетворення аудіо у Мел-спектрограму. 
        # Параметр f_min=150 працює як High-pass фільтр, усуваючи низькочастотний фоновий шум.
        self.mel_transform = T.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=128, n_fft=2048, hop_length=512, f_min=150
        )
        self.db_transform = T.AmplitudeToDB()
        
        if self.augment:
            # Аугментації на рівні сирої хвилі (додавання шуму мікрофона та зміна темпу)
            self.wave_transforms = Compose([
                AddGaussianNoise(min_amplitude=0.01, max_amplitude=0.02, p=0.5),
                TimeStretch(min_rate=0.9, max_rate=1.1, p=0.5) 
            ])
            # SpecAugment: маскування випадкових смуг частот і часу для підвищення робастності моделі
            self.freq_mask = T.FrequencyMasking(freq_mask_param=25) 
            self.time_mask = T.TimeMasking(time_mask_param=40)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self.data_dir / "train_soundscapes" / row['filename']
        
        t_start = int(pd.to_timedelta(row['start']).total_seconds())
        try:
            wav, _ = torchaudio.load(path, frame_offset=t_start * SAMPLE_RATE, num_frames=CHUNK_LEN)
            if wav.shape[0] > 1: wav = wav.mean(0, keepdim=True)
            if wav.shape[1] < CHUNK_LEN:
                wav = torch.nn.functional.pad(wav, (0, CHUNK_LEN - wav.shape[1]))
        except Exception:
            wav = torch.zeros((1, CHUNK_LEN))
            
        if self.augment:
            wav_np = wav.numpy().squeeze()
            wav_np = self.wave_transforms(samples=wav_np, sample_rate=SAMPLE_RATE)
            wav = torch.tensor(wav_np, dtype=torch.float32).unsqueeze(0)
            
        spec = self.mel_transform(wav)
        spec = self.db_transform(spec)
        
        if self.augment:
            # Застосування кількох вузьких масок замість однієї широкої
            for _ in range(3):
                spec = self.freq_mask(spec)
                spec = self.time_mask(spec)
            
        # Нормалізація значень спектрограми у діапазон [0, 1]
        spec = (spec - spec.min()) / (spec.max() - spec.min() + 1e-6)
        
        # Multi-label кодування (One-Hot)
        birds = [b.strip() for b in str(row['primary_label']).split(';')]
        y_vec = np.zeros(NUM_ACTIVE_BIRDS, dtype=np.float32)
        for b in birds:
            if b in ACTIVE_BIRD_SPECIES:
                y_vec[ACTIVE_BIRD_SPECIES.index(b)] = 1.0
                
        return spec, torch.tensor(y_vec)

# --- АРХІТЕКТУРА МОДЕЛІ (PYTORCH LIGHTNING) ---
class BirdLightningModel(pl.LightningModule):
    def __init__(self, model_name, num_classes, lr=1e-2, use_mixup=False):
        super().__init__()
        self.save_hyperparameters()
        self.model = timm.create_model(model_name, pretrained=True, in_chans=1, num_classes=num_classes)
        self.use_mixup = use_mixup
        
        self.criterion = FocalLoss(alpha=0.25, gamma=2.0)
        
        self.train_auroc = torchmetrics.classification.MultilabelAUROC(num_labels=num_classes, average='macro')
        self.val_auroc = torchmetrics.classification.MultilabelAUROC(num_labels=num_classes, average='macro')

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        
        # Метод регуляризації Mixup: лінійна інтерполяція двох випадкових спектрограм
        # та їх міток. Допомагає уникнути жорсткого запам'ятовування трейну (overfitting).
        if self.use_mixup and np.random.rand() < 0.5:
            lam = np.random.beta(0.4, 0.4)
            rand_index = torch.randperm(x.size(0)).to(x.device)
            x = lam * x + (1.0 - lam) * x[rand_index]
            y = lam * y + (1.0 - lam) * y[rand_index]

        logits = self(x)
        loss = self.criterion(logits, y)
        
        self.train_auroc(torch.sigmoid(logits), y.long())
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_auroc", self.train_auroc, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        logits = self(x)
        loss = self.criterion(logits, y)
        
        self.val_auroc(torch.sigmoid(logits), y.long())
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_auroc", self.val_auroc, on_step=False, on_epoch=True, prog_bar=True)

    def configure_optimizers(self):
        # Оптимізатор з Weight Decay (L2 регуляризація) для обмеження росту ваг
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=0.01)
        
        # Cosine Annealing плавно зменшує learning rate до мінімуму (eta_min),
        # дозволяючи моделі точніше зійтися до локального мінімуму в кінці тренування
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=self.trainer.max_epochs, eta_min=1e-6
        )
        return {"optimizer": optimizer, "lr_scheduler": scheduler}

# --- ОСНОВНИЙ ПАЙПЛАЙН ---
def train_pipeline():
    unique_files = df_labels['filename'].unique()
    np.random.shuffle(unique_files)
    
    # Розбиття на train/val за унікальними файлами, 
    split_idx = int(len(unique_files) * 0.8)
    train_df = df_labels[df_labels['filename'].isin(unique_files[:split_idx])]
    val_df = df_labels[df_labels['filename'].isin(unique_files[split_idx:])]
    
    print(f"Записів для навчання: {len(train_df)}, для валідації: {len(val_df)}")
    
    val_loader = DataLoader(BirdCLEFDataset(val_df, DATA_DIR, augment=False), batch_size=32, num_workers=2, shuffle=False)

    configs = [
        {"id": "step_1_resnet_base", "arch": "resnet34", "aug": False, "mixup": False},
        {"id": "step_2_efficientnet_base", "arch": "efficientnet_b0", "aug": True, "mixup": False},
        {"id": "step_3_efficientnet_aug_mixup", "arch": "efficientnet_b0", "aug": True, "mixup": True},
    ]
    
    results = {}
    for cfg in configs:
        print(f"\n>>> Запуск навчання: {cfg['id']}")
        train_loader = DataLoader(BirdCLEFDataset(train_df, DATA_DIR, augment=cfg["aug"]), 
                                  batch_size=32, num_workers=2, shuffle=True)
        
        model = BirdLightningModel(model_name=cfg["arch"], num_classes=NUM_ACTIVE_BIRDS, use_mixup=cfg["mixup"])
        
        checkpoint_cb = ModelCheckpoint(
            dirpath=f"checkpoints/{cfg['id']}", monitor="val_auroc", mode="max", filename="best"
        )
        
        # Early Stopping зупиняє навчання, якщо валідаційна метрика не росте (patience=3),
        # що є головним запобіжником від overfitting-у
        early_stop_cb = EarlyStopping(
            monitor="val_auroc", patience=3, mode="max", verbose=True
        )
        
        trainer = pl.Trainer(
            max_epochs=15,
            accelerator="gpu" if torch.cuda.is_available() else "cpu", 
            devices=1, 
            callbacks=[checkpoint_cb, early_stop_cb], 
            precision="16-mixed",
            log_every_n_steps=10
        )
        
        trainer.fit(model, train_loader, val_loader)
        
        best_ckpt_path = checkpoint_cb.best_model_path
        best_model = BirdLightningModel.load_from_checkpoint(best_ckpt_path, model_name=cfg["arch"], num_classes=NUM_ACTIVE_BIRDS)
        save_path = f"{cfg['id']}_weights.pth"
        torch.save(best_model.model.state_dict(), save_path)
        print(f"--- Ваги {cfg['id']} збережено: {save_path} ---")
        results[cfg['id']] = save_path
        
        del model, trainer, train_loader, best_model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    return results

if __name__ == "__main__":
    best_paths = train_pipeline()
    print("\nУсі етапи навчання завершено! Файли ваг знаходяться у робочій директорії:")
    for name, path in best_paths.items():
        print(f" - {name}: {path}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 15.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: soxr
    Found existing installation: soxr 1.0.0
    Uninstalling soxr-1.0.0:
      Successfully uninstalled soxr-1.0.0


Seed set to 42


Птахів у тренувальній вибірці: 75 / 234
Записів для навчання: 1148, для валідації: 330

>>> Запуск навчання: step_1_resnet_base


model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
E0000 00:00:1776710244.583272      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776710244.648326      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776710245.175764      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776710245.175801      55 computation_placer.cc:177] computation placer already registered. Please check l

...

┏━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name        ┃ Type            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model       │ EfficientNet    │  4.1 M │ train │     0 │
│ 1 │ criterion   │ FocalLoss       │      0 │ train │     0 │
│ 2 │ train_auroc │ MultilabelAUROC │      0 │ train │     0 │
│ 3 │ val_auroc   │ MultilabelAUROC │      0 │ train │     0 │
└───┴─────────────┴─────────────────┴────────┴───────┴───────┘

Trainable params: 4.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 4.1 M                                                                                                
Total estimated model params size (MB): 16                                                                         
Modules in train mode: 340                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

Metric val_auroc improved. New best score: 0.343
Metric val_auroc improved by 0.010 >= min_delta = 0.0. New best score: 0.353
Metric val_auroc improved by 0.026 >= min_delta = 0.0. New best score: 0.379
Metric val_auroc improved by 0.006 >= min_delta = 0.0. New best score: 0.384
Monitored metric val_auroc did not improve in the last 3 records. Best score: 0.384. Signaling Trainer to stop.


--- Ваги step_3_efficientnet_aug_mixup збережено: step_3_efficientnet_aug_mixup_weights.pth ---

Усі етапи навчання завершено! Файли ваг знаходяться у робочій директорії:
 - step_1_resnet_base: step_1_resnet_base_weights.pth
 - step_2_efficientnet_base: step_2_efficientnet_base_weights.pth
 - step_3_efficientnet_aug_mixup: step_3_efficientnet_aug_mixup_weights.pth
